# Exploration of international match results

## 1. Data
I load the raw tables from DuckDB and check their content.

In [11]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.database import query

query("SELECT * FROM raw_results ORDER BY date DESC LIMIT 5")

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,2026-08-26,Vietnam,Thailand,2,2,ASEAN Championship,Hanoi,Vietnam,False
1,2026-08-22,Thailand,Vietnam,0,2,ASEAN Championship,Bangkok,Thailand,False
2,2026-08-19,Vietnam,Malaysia,2,0,ASEAN Championship,Hanoi,Vietnam,False
3,2026-08-18,Thailand,Singapore,1,2,ASEAN Championship,Bangkok,Thailand,False
4,2026-08-16,Malaysia,Vietnam,0,2,ASEAN Championship,Kuala Lumpur,Malaysia,False


In [12]:
query("""
    SELECT
        COUNT(*) AS n_matches,
        MIN(date) AS first_match,
        MAX(date) AS last_match,
        COUNT(DISTINCT home_team) AS n_home_teams,
        ROUND(AVG(neutral::INTEGER), 3) AS share_neutral
    FROM raw_results
""")

,n_matches,first_match,last_match,n_home_teams,share_neutral
0,49547,1872-11-30,2026-08-26,328,0.266


The columns are in the right order and the last match is on 2026-08-26, matching the pinned dataset commit.
The data has 328 distinct home teams, more than the 211 FIFA members, so some non-FIFA teams need a decision.
About 27% of matches are played on neutral ground, so home advantage must depend on the `neutral` flag.

## 2. Coverage over time and competitions

In [13]:
by_decade = query("""
    SELECT
        (YEAR(date) // 10) * 10 AS decade,
        COUNT(*) AS n_matches,
        COUNT(DISTINCT home_team) AS n_teams
    FROM raw_results
    GROUP BY decade
    ORDER BY decade
""")
by_decade

,decade,n_matches,n_teams
0,1870,13,3
1,1880,55,5
2,1890,59,4
3,1900,138,19
4,1910,330,33
5,1920,831,68
6,1930,1079,76
7,1940,833,95
8,1950,1651,124
9,1960,2971,152


In [14]:
query("""
    SELECT tournament, COUNT(*) AS n_matches
    FROM raw_results
    WHERE date >= '2000-01-01'
    GROUP BY tournament
    ORDER BY n_matches DESC
    LIMIT 20
""")

,tournament,n_matches
0,Friendly,8445
1,FIFA World Cup qualification,5840
2,UEFA Euro qualification,1532
3,African Cup of Nations qualification,1416
4,UEFA Nations League,658
5,AFC Asian Cup qualification,530
6,African Cup of Nations,525
7,FIFA World Cup,488
8,CONCACAF Nations League,422
9,Gold Cup,359


Before 1950 the data is sparse. The volume grows fast from the 1990s.
Friendlies are a third of matches since 2000, and they carry less information than competitive games.
Some tournaments are non-FIFA (Island Games) and some names changed over time (AFF / ASEAN Championship).

I compute Elo ratings on the full history so they have time to converge (burn-in).
I train and evaluate the models on matches from 2000 onward.
I keep friendlies but give them a lower weight in the Elo update.

## 3. Non-FIFA teams
The dataset does not flag FIFA members. I use a proxy: a team that never played a World Cup qualifier since 2000 is likely not a FIFA member.

In [15]:
non_fifa = query("""
    WITH team_matches AS (
        SELECT home_team AS team, tournament, date FROM raw_results
        UNION ALL
        SELECT away_team AS team, tournament, date FROM raw_results
    ),
    qualifier_teams AS (
        SELECT DISTINCT team
        FROM team_matches
        WHERE tournament = 'FIFA World Cup qualification'
          AND date >= '2000-01-01'
    )
    SELECT team, COUNT(*) AS n_matches, MAX(date) AS last_match
    FROM team_matches
    WHERE date >= '2000-01-01'
      AND team NOT IN (SELECT team FROM qualifier_teams)
    GROUP BY team
    ORDER BY n_matches DESC
""")

n_qualifier_teams = query("""
    SELECT COUNT(DISTINCT team) FROM (
        SELECT home_team AS team FROM raw_results
        WHERE tournament = 'FIFA World Cup qualification' AND date >= '2000-01-01'
        UNION
        SELECT away_team FROM raw_results
        WHERE tournament = 'FIFA World Cup qualification' AND date >= '2000-01-01'
    )
""").iloc[0, 0]

print(f"Teams in World Cup qualifiers since 2000: {n_qualifier_teams}")
print(f"Other teams active since 2000: {len(non_fifa)}")
non_fifa.head(30)

Teams in World Cup qualifiers since 2000: 211
Other teams active since 2000: 111


,team,n_matches,last_match
0,Martinique,158,2026-03-29
1,Guadeloupe,143,2025-06-24
2,French Guiana,109,2024-11-19
3,Jersey,85,2025-07-18
4,Zanzibar,81,2025-01-13
5,Guernsey,78,2023-07-13
6,Saint Martin,64,2026-03-29
7,Padania,48,2026-06-06
8,Greenland,48,2026-06-06
9,Alderney,47,2022-04-23


In [16]:
non_fifa.tail(20)

,team,n_matches,last_match
91,Quebec,3,2013-06-27
92,Hmong,3,2023-08-08
93,Biafra,3,2022-05-25
94,Chechnya,3,2006-02-18
95,Yoruba Nation,2,2022-05-23
96,Canary Islands,2,2007-12-29
97,Aymara,2,2022-06-19
98,West Papua,2,2022-11-13
99,Délvidék,2,2016-08-03
100,Marshall Islands,2,2025-08-16


## 4. Outcomes and home advantage
I look at the share of home wins, draws and away wins since 2000, split by neutral ground.

In [17]:
query("""
    SELECT
        neutral,
        COUNT(*) AS n_matches,
        ROUND(AVG((home_score > away_score)::INTEGER), 3) AS home_win,
        ROUND(AVG((home_score = away_score)::INTEGER), 3) AS draw,
        ROUND(AVG((home_score < away_score)::INTEGER), 3) AS away_win,
        ROUND(AVG(home_score), 2) AS home_goals,
        ROUND(AVG(away_score), 2) AS away_goals
    FROM clean_results
    WHERE date >= '2000-01-01'
    GROUP BY neutral
""")

,neutral,n_matches,home_win,draw,away_win,home_goals,away_goals
0,False,17511,0.506,0.234,0.260,1.67,1.01
1,True,6449,0.407,0.255,0.339,1.42,1.26


At home, teams win 51% of matches and score 0.67 more goals than the visitors, so home advantage is large.
Draws are about a quarter of matches, and they are never the most likely outcome, so I evaluate probabilities, not accuracy.
On neutral ground the listed home team still wins more often (41% vs 34%). The listed home team is likely the seeded, stronger side, so I apply home advantage only when `neutral` is false. I will check this with Elo ratings.

## 5. Target
I code the result as H (home win), D (draw) or A (away win).

In [18]:
query("""
    SELECT
        result,
        COUNT(*) AS n_matches,
        ROUND(COUNT(*) / SUM(COUNT(*)) OVER (), 3) AS share
    FROM clean_results
    WHERE date >= '2000-01-01'
    GROUP BY result
    ORDER BY result
""")

,result,n_matches,share
0,A,6731,0.281
1,D,5738,0.239
2,H,11491,0.480


Since 2000: 48% home wins, 24% draws, 28% away wins, over 23,960 matches.
The classes are imbalanced but not extreme. I do not rebalance them, because I want calibrated probabilities.

## 6. Extra time
Scores in the results file include extra time. I use the goalscorer file to remove extra-time goals and keep the score at the end of regulation time, stoppage time included.
Stoppage-time goals are coded as minute 90, even the long 2022 and 2026 World Cup stoppages, so they are kept.
A match went to extra time if removing goals after minute 90 gives a draw, or if a goal was scored after minute 100.
A shootout match is coded as a draw, which matches the play.

In [19]:
query("""
    SELECT date, home_team, away_team,
           home_score_final, away_score_final,
           home_score, away_score, result
    FROM clean_results
    WHERE extra_time AND date >= '2026-06-01'
    ORDER BY date
""")

,date,home_team,away_team,home_score_final,away_score_final,home_score,away_score,result
0,2026-07-01,Belgium,Senegal,3,2,2,2,D
1,2026-07-03,Argentina,Cape Verde,3,2,1,1,D
2,2026-07-11,Argentina,Switzerland,3,1,1,1,D
3,2026-07-11,Norway,England,1,2,1,1,D
4,2026-07-19,Spain,Argentina,1,0,0,0,D


In [ ]:
query("""
    SELECT COUNT(*) AS n_matches, SUM(extra_time::INTEGER) AS n_extra_time
    FROM clean_results
    WHERE date >= '2000-01-01'
""")

The five 2026 World Cup matches with extra time all have a level score at the end of regulation time.
103 matches since 2000 are corrected. Scorer data is incomplete outside major tournaments, and a few old group matches with stoppage time coded after minute 90 may be misread. I note both as limitations.